# 26-тарау. Бидай өнімін болжаймыз: ЖИ ауыл шаруашылығында

*«Жасанды интеллект негіздері» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 26.1. Жобаның мақсаты

## 26.2. 1-қадам. Деректермен танысу

In [ ]:
import numpy as np
import pandas as pd

URL = ("https://raw.githubusercontent.com/amirtemir1996n3-dotcom/"
       "mashina-qalai-uirenedi/main/data/wheat_kz.csv")
df = pd.read_csv(URL)
print(df.shape)
print(df.head(3).to_string())

In [ ]:
print(df.groupby("облыс")[["жауын_шашын", "гумус", "өнім"]].mean().round(1))

## 26.3. 2-қадам. Факторлардың әсерін көру

In [ ]:
import matplotlib.pyplot as plt

cols = ["жауын_шашын", "шілде_темп", "тыңайтқыш", "егу_күні"]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2), sharey=True)
for ax, col in zip(axes, cols):
    ax.scatter(df[col], df["өнім"], s=5, alpha=0.4)
    ax.set_xlabel(col)
axes[0].set_ylabel("өнім, ц/га")
plt.show()

## 26.4. 3-қадам. Облысты санға айналдыру

In [ ]:
X = pd.get_dummies(df.drop(columns=["өнім", "жыл"]),
                   columns=["облыс"], dtype=int)
y = df["өнім"]
print(X.shape)
print(X.filter(like="облыс").head(3))

## 26.5. 4-қадам. Сызықтық модель мен кездейсоқ орман

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)

lin = LinearRegression().fit(X_train, y_train)
forest = RandomForestRegressor(n_estimators=300, min_samples_leaf=3,
                               random_state=42).fit(X_train, y_train)
for name, model in [("Сызықтық", lin), ("Орман", forest)]:
    pred = model.predict(X_test)
    print(f"{name:9} MAE = {mean_absolute_error(y_test, pred):.2f} ц/га, "
          f"R² = {r2_score(y_test, pred):.2f}")

## 26.6. 5-қадам. Қай фактор ең маңызды?

In [ ]:
imp = pd.Series(forest.feature_importances_, index=X.columns)
print(imp.sort_values(ascending=False).round(3))

## 26.7. 6-қадам. «Егер… болса?» талдауы

In [ ]:
for rain in range(100, 401, 50):
    X_what = X_test.copy()
    X_what["жауын_шашын"] = rain
    print(f"{rain} мм → {forest.predict(X_what).mean():.1f} ц/га")

## 26.8. 7-қадам. Құрғақшылық сценарийі

In [ ]:
drought = X_test.copy()
drought["жауын_шашын"] *= 0.6
drought["шілде_темп"] += 2
normal = forest.predict(X_test).mean()
dry = forest.predict(drought).mean()
print(f"Қалыпты жыл: {normal:.1f} ц/га")
print(f"Құрғақшылық: {dry:.1f} ц/га ({dry / normal - 1:+.0%})")

In [ ]:
t_max = X_train["шілде_темп"].max()
r_min = X_train["жауын_шашын"].min()
print(f"Оқу деректеріндегі ең ыстық шілде: {t_max} °C")
print("Сценарийде одан ыстық алқаптар:",
      (drought["шілде_темп"] > t_max).sum())
print(f"Оқу деректеріндегі ең аз жауын: {r_min} мм")
print("Сценарийде одан құрғақ алқаптар:",
      (drought["жауын_шашын"] < r_min).sum())